# Sepang counterfactual: what if Hamilton had started on inters?

2026 Bahrain Grand Prix (held at Sepang, Round 16, 4 Oct 2026). Ferrari started Hamilton from P2 on slicks on a wet track. He finished P3.
This notebook runs the whole analysis end to end. Every number comes from `config.yaml` and the code in `src/`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # the notebook runs from notebooks/; src/ lives at the repo root
import fastf1; fastf1.set_log_level("ERROR")  # hide download chatter

from src.data_loader import load_config, load_session, fact_check

cfg = load_config()
session = load_session(cfg)

## M1: what does the data actually say?

Before modelling anything, check the race story against the timing data. The PRD's rule is that **the data wins**.

In [2]:
fact_check(session, cfg)


=== Event ===
Bahrain Grand Prix | Kuala Lumpur | 2026-10-04 | 55 laps

=== Official classification (top 6 + Ferraris) ===
                     TeamName  GridPosition  Position    Status  gap_s
Abbreviation                                                          
VER           Red Bull Racing           1.0       1.0  Finished  0.000
ANT                  Mercedes           3.0       2.0  Finished  2.307
HAM                   Ferrari           2.0       3.0  Finished  4.919
LEC                   Ferrari           4.0       4.0  Finished  7.258
HAD           Red Bull Racing           8.0       5.0  Finished  8.571
PIA                   McLaren           6.0       6.0  Finished  9.454

=== How the race started (race control) ===
 Lap                                                                                                                         Message
   1                                                                                                                   DELAYED STA

### What the data changed

| PRD / press said | Data says | Why it matters |
|---|---|---|
| Wet race | **No rain fell during the race.** It stopped about 54 min before the start, so the race ran on a drying track | Inter pace improves lap by lap, so a single "inter pace" number is wrong |
| Hamilton P2 → ~P16 on lap 1 | Laps 1–2 were **behind the SC**, then a **standing start**. HAM was P15 after lap 3 (the first racing lap) | The PRD's "lap-1 delta" becomes a lap-3 standing-start delta |
| Long wet phase, then a crossover | Wet phase = **laps 3–8 only**. All the leaders pitted for slicks on **lap 9, under the SC** | The crossover stop was cheap for everyone. The ghost gets the same cheap stop |
| Gamble cost Hamilton the race | He was **83 s** behind after lap 8, but only **6.5 s** after the SC, at P15 | The SC refunded the *time* but not the *track position*. Then he had to pass 10 cars |
| Leclerc P4 | Official P4, but FastF1 timing puts him P17 (+26.8 s) | A per-car timing glitch (+19.5 s from lap 43). We use the official classification for results |

**FastF1 lessons:**
- If the classification API is down, `session.results` quietly falls back to timing-derived positions. `load_session` refuses to continue if that happens.
- Unknown compounds come through as the *string* `'None'`.
- `TrackStatus` doesn't show the SC at the race start.
- Count a lap as SC/VSC only when most cars saw it, because back-markers' laps overlap the leaders' SC.